# React 框架 · 第 05 天

## React Router 记账本项目实战

本篇依据已下载的 React 基础 day04+day05 配套包中的记账本 MD 笔记、PPT 和代码整理。内容覆盖项目环境、路径别名、Mock 接口、Ant Design Mobile 主题、Redux 账单数据、底部 TabBar、月度统计、按日账单列表、图标组件和新增账单功能。

## 一、学习目标

学完本篇后，应能：

- 搭建包含 Redux、React Router、Ant Design Mobile、Axios 和 dayjs 的移动端项目。
- 使用 CRACO 配置 `@/` 指向 `src/`，提高模块导入可读性。
- 使用 json-server 模拟账单接口，先完成前端业务开发。
- 设计 Layout、月度账单、年度账单和新增账单的路由结构。
- 用 Redux Toolkit 请求并保存账单列表。
- 使用 TabBar 完成底部导航和路由切换。
- 按月、按日分组账单并计算支出、收入、结余。
- 完成账单分类图标、收支切换、金额输入和新增账单。

## 二、项目环境与依赖

记账本是一个移动端单页应用，基础依赖如下：

```bash
npx create-react-app react-bill
cd react-bill
npm i @reduxjs/toolkit react-redux
npm i react-router-dom
npm i dayjs classnames
npm i antd-mobile antd-mobile-icons
npm i axios
npm i -D json-server @craco/craco
```

各依赖的职责：

- `@reduxjs/toolkit`、`react-redux`：管理共享账单状态。
- `react-router-dom`：管理页面路由。
- `dayjs`：格式化日期和按月份、日期分组。
- `classnames`：根据状态拼接 CSS 类名。
- `antd-mobile`、`antd-mobile-icons`：提供移动端 UI 组件和图标。
- `axios`：请求账单接口。
- `json-server`：本地模拟后端接口。
- `@craco/craco`：无需直接改 webpack 即可配置路径别名。

## 三、配置 `@/` 路径别名

项目中常用 `@/store`、`@/components` 等写法，把 `@/` 解析为 `src/`，可以避免层级很深的相对路径。

### 1. 安装 CRACO

```bash
npm i -D @craco/craco
```

### 2. 创建 `craco.config.js`

```js
const path = require('path')

module.exports = {
  webpack: {
    alias: {
      '@': path.resolve(__dirname, 'src'),
    },
  },
}
```

### 3. 修改 `package.json` 脚本

```json
{
  "scripts": {
    "start": "craco start",
    "build": "craco build",
    "test": "craco test"
  }
}
```

### 4. 配置编辑器联想

根目录创建 `jsconfig.json`：

```json
{
  "compilerOptions": {
    "baseUrl": "./",
    "paths": {
      "@/*": ["src/*"]
    }
  }
}
```

webpack 解析配置负责项目运行，`jsconfig.json` 负责 VS Code 的路径提示，两者都要配置。

## 四、使用 json-server 模拟接口

前后端分离时，前端可以先通过 Mock 数据开发页面和交互，不必等待真实后端。常见方式包括静态 JSON、Mock 服务、在线 Mock 平台和本地 `json-server`。

### 1. 准备数据

在项目根目录创建 `db.json`：

```json
{
  "ka": [
    {
      "id": 1,
      "type": "pay",
      "money": 35,
      "date": "2023-03-23",
      "useFor": "food"
    }
  ]
}
```

### 2. 添加启动命令

```json
{
  "scripts": {
    "serve": "json-server ./db.json --port 8888"
  }
}
```

启动：

```bash
npm run serve
```

接口地址为 `http://localhost:8888/ka`。Mock 数据结构要尽量接近未来真实接口，后续切换真实服务时只替换请求地址和异常处理。

## 五、整体路由与目录设计

记账本可以设计为两个一级路由：

- `Layout`：公共布局，包含内容出口和底部 TabBar。
- `New`：新增账单页。

Layout 下再放两个二级路由：

- `Month`：月度账单。
- `Year`：年度账单。

```text
/                 → Layout + Month（默认）
/month            → 月度账单
/year             → 年度账单
/new              → 新增账单
*                 → 404
```

```jsx
const router = createBrowserRouter([
  {
    path: '/',
    element: <Layout />,
    children: [
      { index: true, element: <Navigate to="/month" replace /> },
      { path: 'month', element: <Month /> },
      { path: 'year', element: <Year /> },
    ],
  },
  { path: '/new', element: <New /> },
  { path: '*', element: <NotFound /> },
])
```

公共页面壳由 Layout 负责，具体账单页面通过 `Outlet` 渲染，新增账单因为交互结构独立，可以作为单独一级路由。

## 六、Ant Design Mobile 主题定制

Ant Design Mobile 支持通过 CSS 变量调整主题。记账本使用绿色作为主色：

```css
:root:root {
  --adm-color-primary: rgb(105, 174, 120);
}
```

### 全局定制

在全局样式中覆盖变量，所有组件都使用统一主题。

### 局部定制

在组件或局部容器上覆盖变量，只影响当前区域：

```css
.keepAccounts {
  --adm-color-primary: #69ae78;
}
```

全局变量适合品牌色、字体等整体规范；局部变量适合记账页背景、按钮和分类选中态等页面专属样式。

## 七、Redux 管理账单列表

账单列表需要被月度账单、年度账单和统计组件共同使用，适合放在 Redux 中。

```js
// store/modules/billStore.js
import { createSlice } from '@reduxjs/toolkit'
import axios from 'axios'

const billStore = createSlice({
  name: 'bill',
  initialState: {
    billList: [],
  },
  reducers: {
    setBillList(state, action) {
      state.billList = action.payload
    },
    addBill(state, action) {
      state.billList.push(action.payload)
    },
  },
})

export const { setBillList, addBill } = billStore.actions

export const getBillList = () => async dispatch => {
  const response = await axios.get('http://localhost:8888/ka')
  dispatch(setBillList(response.data))
}

export default billStore.reducer
```

组合并注入：

```js
// store/index.js
import { configureStore } from '@reduxjs/toolkit'
import billReducer from './modules/billStore'

export default configureStore({
  reducer: { bill: billReducer },
})
```

```jsx
<Provider store={store}>
  <RouterProvider router={router} />
</Provider>
```

## 八、Layout 与底部 TabBar

Layout 负责放置页面出口和底部导航，并在首次进入时加载账单：

```jsx
import { TabBar } from 'antd-mobile'
import { useEffect } from 'react'
import { Outlet, useNavigate, useLocation } from 'react-router-dom'
import { useDispatch } from 'react-redux'
import { getBillList } from '@/store/modules/billStore'
import { BillOutline, CalculatorOutline, AddCircleOutline } from 'antd-mobile-icons'

const tabs = [
  { key: '/month', title: '月度账单', icon: <BillOutline /> },
  { key: '/new', title: '记账', icon: <AddCircleOutline /> },
  { key: '/year', title: '年度账单', icon: <CalculatorOutline /> },
]

function Layout() {
  const dispatch = useDispatch()
  const navigate = useNavigate()
  const location = useLocation()

  useEffect(() => {
    dispatch(getBillList())
  }, [dispatch])

  return (
    <div className="layout">
      <main className="container"><Outlet /></main>
      <footer className="footer">
        <TabBar
          activeKey={location.pathname}
          onChange={key => navigate(key)}
        >
          {tabs.map(item => (
            <TabBar.Item key={item.key} icon={item.icon} title={item.title} />
          ))}
        </TabBar>
      </footer>
    </div>
  )
}
```

底部布局可用固定定位：

```css
.layout .container {
  position: fixed;
  inset: 0 0 50px;
  overflow-y: auto;
}
.layout .footer {
  position: fixed;
  bottom: 0;
  width: 100%;
}
```

`activeKey` 要和当前路径同步，否则页面跳转后 Tab 高亮可能停留在旧位置。

## 九、月度账单：月份选择和统计区域

月度账单页面包括导航栏、月份选择、支出/收入/结余统计和账单列表。月份选择器用 `DatePicker`，通过状态控制打开和关闭：

```jsx
import dayjs from 'dayjs'
import classNames from 'classnames'
import { DatePicker, NavBar } from 'antd-mobile'
import { useState } from 'react'

function MonthHeader() {
  const [dateVisible, setDateVisible] = useState(false)
  const [currentMonth, setCurrentMonth] = useState(() =>
    dayjs().format('YYYY-MM')
  )

  const dateConfirm = date => {
    setDateVisible(false)
    setCurrentMonth(dayjs(date).format('YYYY-MM'))
  }

  return (
    <div className="header">
      <div className="date" onClick={() => setDateVisible(true)}>
        <span>{currentMonth} 账单</span>
        <span className={classNames('arrow', dateVisible && 'expand')} />
      </div>
      <DatePicker
        title="记账日期"
        precision="month"
        visible={dateVisible}
        max={new Date()}
        onConfirm={dateConfirm}
        onCancel={() => setDateVisible(false)}
      />
    </div>
  )
}
```

月份选择、箭头样式和弹框显示都由同一个 `dateVisible` 状态驱动，避免只改了弹框却忘记同步图标。

## 十、按月分组并计算总览

先从 Redux 读取账单，再用 `dayjs` 按月份分组：

```jsx
import _ from 'lodash'
import dayjs from 'dayjs'
import { useMemo } from 'react'
import { useSelector } from 'react-redux'

const billList = useSelector(state => state.bill.billList)
const monthGroup = useMemo(() => {
  return _.groupBy(billList, item =>
    dayjs(item.date).format('YYYY-MM')
  )
}, [billList])
```

根据当前月份取得账单并计算统计值：

```jsx
const currentMonthList = monthGroup[currentMonth] || []

const overview = useMemo(() => {
  const pay = currentMonthList
    .filter(item => item.type === 'pay')
    .reduce((sum, item) => sum + Math.abs(Number(item.money)), 0)
  const income = currentMonthList
    .filter(item => item.type === 'income')
    .reduce((sum, item) => sum + Math.abs(Number(item.money)), 0)

  return { pay, income, balance: income - pay }
}, [currentMonthList])
```

金额参与计算前统一转为数字，并明确“支出、收入、结余”的含义。如果接口使用负数表示支出，应在统计层用 `Math.abs` 转成展示金额，再用 `income - pay` 计算余额。

## 十一、日账单组件与按日分组

当前月份拿到后，再按 `YYYY-MM-DD` 分组：

```jsx
const dayGroup = useMemo(() => {
  const group = _.groupBy(currentMonthList, item =>
    dayjs(item.date).format('YYYY-MM-DD')
  )

  return {
    dayKeys: Object.keys(group).sort().reverse(),
    group,
  }
}, [currentMonthList])
```

页面遍历分组结果，把当天的日期和账单数组传给 `DailyBill`：

```jsx
{dayGroup.dayKeys.map(dayKey => (
  <DailyBill
    key={dayKey}
    date={dayKey}
    billList={dayGroup.group[dayKey]}
  />
))}
```

`DailyBill` 自己负责计算当天的支出、收入和结余，这样月度页面只负责分组和组合，日组件负责日级展示。

## 十二、单日统计与账单列表

`DailyBill` 中使用 `useMemo` 计算当天统计：

```jsx
function DailyBill({ date, billList }) {
  const result = useMemo(() => {
    const pay = billList
      .filter(item => item.type === 'pay')
      .reduce((sum, item) => sum + Math.abs(Number(item.money)), 0)
    const income = billList
      .filter(item => item.type === 'income')
      .reduce((sum, item) => sum + Math.abs(Number(item.money)), 0)

    return { pay, income, balance: income - pay }
  }, [billList])

  return (
    <section className="dailyBill">
      <header>
        <span>{date}</span>
        <span>支出 {result.pay.toFixed(2)}</span>
        <span>收入 {result.income.toFixed(2)}</span>
        <span>结余 {result.balance.toFixed(2)}</span>
      </header>
      <div className="billList">
        {billList.map(item => (
          <div className="bill" key={item.id}>
            <Icon type={item.useFor} />
            <span>{billTypeToName[item.useFor]}</span>
            <span className={item.type}>
              {Math.abs(Number(item.money)).toFixed(2)}
            </span>
          </div>
        ))}
      </div>
    </section>
  )
}
```

列表元素使用账单 id 作为 key；金额展示统一格式化为两位小数；支出和收入通过类型类名呈现不同颜色。

## 十三、账单分类字典与 Icon 组件

新增账单和历史账单都需要把 `useFor` 类型转换成人能读懂的名称，可以准备分类数据并生成映射表：

```js
export const billListData = {
  pay: [
    {
      type: 'foods',
      name: '餐饮',
      list: [
        { type: 'food', name: '餐费' },
        { type: 'drinks', name: '酒水饮料' },
      ],
    },
  ],
  income: [
    {
      type: 'professional',
      name: '职业收入',
      list: [{ type: 'salary', name: '工资' }],
    },
  ],
}

export const billTypeToName = Object.keys(billListData).reduce((result, key) => {
  billListData[key].forEach(group => {
    group.list.forEach(item => {
      result[item.type] = item.name
    })
  })
  return result
}, {})
```

图标组件只接收类型，根据类型拼接静态资源地址：

```jsx
const BASE_URL =
  'https://yjy-teach-oss.oss-cn-beijing.aliyuncs.com/reactbase/ka/'

function Icon({ type, className = '' }) {
  return (
    <img
      className={className}
      src={`${BASE_URL}${type}.svg`}
      alt=""
      width="20"
      height="20"
    />
  )
}
```

把 URL、类型映射和视觉组件分开，账单列表只需要传 `type`，后续更换图标来源时改动集中。

## 十四、展开和收起每日账单

日账单默认可以只展示统计，点击箭头后展开明细：

```jsx
function DailyBill({ date, billList }) {
  const [visible, setVisible] = useState(false)

  return (
    <section className={classNames('dailyBill', visible && 'expand')}>
      <header onClick={() => setVisible(prev => !prev)}>
        <span>{date}</span>
        <span className={classNames('arrow', visible && 'expand')} />
      </header>
      <div
        className="billList"
        style={{ display: visible ? 'block' : 'none' }}
      >
        {billList.map(item => (
          <div key={item.id}>{billTypeToName[item.useFor]}</div>
        ))}
      </div>
    </section>
  )
}
```

也可以用 CSS 类控制显示：`.dailyBill.expand .billList { display: block; }`。交互状态属于单个日账单组件，不需要放到 Redux。

## 十五、新增账单页面：结构

新增页面包括：

- 支出/收入切换。
- 日期选择。
- 金额输入。
- 账单分类选择。
- 保存按钮。

```jsx
import { Button, DatePicker, Input, NavBar } from 'antd-mobile'
import { useNavigate } from 'react-router-dom'

function New() {
  const navigate = useNavigate()
  return (
    <div className="keepAccounts">
      <NavBar onBack={() => navigate(-1)}>记一笔</NavBar>
      <div className="header">
        <div className="kaType">
          <Button shape="rounded">支出</Button>
          <Button shape="rounded">收入</Button>
        </div>
        <div className="kaForm">
          <DatePicker title="记账日期" max={new Date()} />
          <Input placeholder="0.00" type="number" />
        </div>
      </div>
      <Button className="save">保 存</Button>
    </div>
  )
}
```

先搭静态结构和样式，再逐步接入状态和事件，可以降低一次性处理布局、数据和交互的复杂度。

## 十六、新增账单：收支与分类切换

用 `billType` 区分支出和收入，用它作为 `billListData` 的 key，分类列表会自动适配：

```jsx
const [billType, setBillType] = useState('pay')
const [useFor, setUseFor] = useState('')

<div className="kaType">
  <Button
    className={classNames(billType === 'pay' && 'selected')}
    onClick={() => { setBillType('pay'); setUseFor('') }}
  >
    支出
  </Button>
  <Button
    className={classNames(billType === 'income' && 'selected')}
    onClick={() => { setBillType('income'); setUseFor('') }}
  >
    收入
  </Button>
</div>

{billListData[billType].map(group => (
  <div className="kaType" key={group.type}>
    <div className="title">{group.name}</div>
    <div className="list">
      {group.list.map(item => (
        <div
          key={item.type}
          className={classNames('item', useFor === item.type && 'selected')}
          onClick={() => setUseFor(item.type)}
        >
          <Icon type={item.type} />
          <div>{item.name}</div>
        </div>
      ))}
    </div>
  </div>
))}
```

切换支出/收入时清空旧分类，避免保留一个不属于当前类型的 `useFor`。

## 十七、新增一笔并写入 Redux

收集金额、日期和分类后，生成一条账单提交给 Redux：

```jsx
const [money, setMoney] = useState('')
const [date, setDate] = useState(new Date())
const [billType, setBillType] = useState('pay')
const [useFor, setUseFor] = useState('')
const dispatch = useDispatch()

const saveBill = () => {
  const amount = Number(money)
  if (!amount || !useFor) return

  const data = {
    type: billType,
    money: amount,
    date: date.toISOString(),
    useFor,
  }

  dispatch(addBill(data))
  navigate('/month')
}
```

若项目约定支出使用负数、收入使用正数，可以在保存时统一处理；但统计和展示必须明确转换规则，避免出现支出负负得正。更稳妥的做法是保存正数金额和 `type`，统计时按类型计算。

保存前要做最基本校验：金额大于 0、已选择分类、日期有效。保存成功后返回月度账单，列表会因 Redux 状态更新自动出现新记录。

## 十八、项目中的数据流

记账本完整的数据流可以画成：

```text
Layout 首次加载
  ↓ dispatch(getBillList())
Redux bill.billList
  ↓ useSelector
Month 按月分组、计算总览
  ↓
DailyBill 按日分组、渲染明细

New 页面填写表单
  ↓ dispatch(addBill(data))
Redux 更新 billList
  ↓
月度和年度页面自动重新渲染
```

页面局部状态（日期弹层、每日明细展开、当前表单输入）由组件 `useState` 管理；账单列表这种跨页面共享数据由 Redux 管理；路由状态由 React Router 管理。三类状态边界清楚，维护成本会更低。

## 十九、常见问题与排查顺序

1. **`@/` 导入失败**：检查 CRACO 是否安装、脚本是否改成 `craco`，并确认 `jsconfig.json` 配置正确。
2. **Mock 接口请求失败**：确认 json-server 已启动、端口为 8888，访问 `/ka` 是否能返回 JSON。
3. **TabBar 高亮不正确**：让 `activeKey` 使用 `location.pathname`，切换时调用 `navigate`。
4. **月度账单为空**：确认账单日期格式能被 dayjs 解析，分组 key 与当前月份格式一致。
5. **支出统计符号错误**：统一金额存储规则，统计时不要重复取负号。
6. **分类名称显示 undefined**：检查 `useFor` 是否存在于 `billTypeToName` 映射。
7. **选择收入后仍高亮旧分类**：切换 `billType` 时重置 `useFor`。
8. **保存后页面没有新账单**：确认 reducer 已注册、dispatch 的 action 正确，列表使用的是 Redux 中的最新数据。
9. **移动端页面被底部 TabBar 遮挡**：给内容容器预留底部高度并设置滚动区域。

建议按“接口 → Redux → 页面选择器 → 分组计算 → 组件渲染”的链路排查，而不是先修改样式。

## 二十、本日总结

- 用 CRACO 和 jsconfig 配置 `@/` 路径别名，提升项目结构可读性。
- 用 json-server 模拟账单接口，先完成前端功能。
- Layout 负责公共布局和 TabBar，Month、Year、New 分别负责业务页面。
- Redux 保存账单列表，页面使用 `useSelector` 读取并通过 `dispatch` 更新。
- 月度账单按月分组，再按日分组，分别由月度统计和 DailyBill 展示。
- Ant Design Mobile 提供移动端导航、按钮、输入框和日期选择器。
- 新增账单页面通过收支类型、金额、日期和分类生成新记录。
- 组件状态、路由状态和全局业务状态各司其职，项目更易维护。

至此，React Router 记账本的基础页面、数据流和核心交互已经串联完成。